# Experiment 7: Text Analytics

**Aim:**
1. Apply document preprocessing: **Tokenization, POS Tagging, Stop Words Removal, Stemming, Lemmatization**
2. Create document representation using **TF-IDF** (Term Frequency - Inverse Document Frequency)

> All code uses only built-in Python and pre-installed libraries (re, sklearn, pandas). No extra installation needed.

### Step 1: Import Libraries

In [1]:
import re
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

print('Libraries imported!')

Libraries imported!


### Step 2: Sample Document

In [2]:
document = """
Natural Language Processing is a fascinating field of Artificial Intelligence.
It helps computers understand interpret and generate human language.
Text mining and machine learning are used together to extract useful information.
The running dogs are barking loudly near the tall buildings.
"""

print('Sample Document:')
print(document)

Sample Document:

Natural Language Processing is a fascinating field of Artificial Intelligence.
It helps computers understand interpret and generate human language.
Text mining and machine learning are used together to extract useful information.
The running dogs are barking loudly near the tall buildings.



### Step 3: Tokenization
**Tokenization** splits text into individual words (tokens).
We use Python's built-in `re` module — no installation needed.

In [3]:
# Extract only alphabetic words and convert to lowercase
tokens = re.findall(r'\b[a-zA-Z]+\b', document.lower())

print('Tokens:')
print(tokens)
print(f'\nTotal Tokens: {len(tokens)}')

Tokens:
['natural', 'language', 'processing', 'is', 'a', 'fascinating', 'field', 'of', 'artificial', 'intelligence', 'it', 'helps', 'computers', 'understand', 'interpret', 'and', 'generate', 'human', 'language', 'text', 'mining', 'and', 'machine', 'learning', 'are', 'used', 'together', 'to', 'extract', 'useful', 'information', 'the', 'running', 'dogs', 'are', 'barking', 'loudly', 'near', 'the', 'tall', 'buildings']

Total Tokens: 41


### Step 4: POS Tagging (Part-of-Speech Tagging)
**POS Tagging** assigns a grammatical label to each word: Noun, Verb, Adjective, etc.

We use a simple rule-based approach:
- Words ending in `ing` → Verb (VBG)
- Words ending in `ed` → Past Verb (VBD)
- Words ending in `ly` → Adverb (RB)
- Words ending in `tion`, `ment`, `ness` → Noun (NN)
- Everything else → Noun (NN) by default

In [4]:
def simple_pos_tag(word):
    if word.endswith('ing'):
        return 'VBG'  # Verb, gerund
    elif word.endswith('ed'):
        return 'VBD'  # Verb, past tense
    elif word.endswith('ly'):
        return 'RB'   # Adverb
    elif word.endswith(('tion', 'ment', 'ness', 'ity')):
        return 'NN'   # Noun
    elif word.endswith('al') or word.endswith('ful') or word.endswith('ous'):
        return 'JJ'   # Adjective
    else:
        return 'NN'   # Default: Noun

pos_tags = [(word, simple_pos_tag(word)) for word in tokens]

print('POS Tags:')
print(f'{"Word":<25} Tag')
print('-' * 35)
for word, tag in pos_tags:
    print(f'{word:<25} {tag}')

POS Tags:
Word                      Tag
-----------------------------------
natural                   JJ
language                  NN
processing                VBG
is                        NN
a                         NN
fascinating               VBG
field                     NN
of                        NN
artificial                JJ
intelligence              NN
it                        NN
helps                     NN
computers                 NN
understand                NN
interpret                 NN
and                       NN
generate                  NN
human                     NN
language                  NN
text                      NN
mining                    VBG
and                       NN
machine                   NN
learning                  VBG
are                       NN
used                      VBD
together                  NN
to                        NN
extract                   NN
useful                    JJ
information               NN
the                 

### Step 5: Stop Words Removal
**Stop words** are common words (like 'the', 'is', 'and') that don't carry much meaning.
We remove them to focus on important words.

In [5]:
# Hardcoded list of common English stop words
stop_words = {
    'the', 'is', 'a', 'an', 'and', 'or', 'in', 'of', 'to', 'it',
    'for', 'on', 'are', 'was', 'be', 'with', 'that', 'this', 'from',
    'by', 'at', 'as', 'has', 'not', 'but', 'its', 'into', 'near',
    'used', 'helps', 'together', 'also'
}

# Keep only words NOT in the stop words list
filtered_tokens = [word for word in tokens if word not in stop_words]

print('Tokens after Stop Words Removal:')
print(filtered_tokens)
print(f'\nBefore: {len(tokens)} tokens  |  After: {len(filtered_tokens)} tokens')

Tokens after Stop Words Removal:
['natural', 'language', 'processing', 'fascinating', 'field', 'artificial', 'intelligence', 'computers', 'understand', 'interpret', 'generate', 'human', 'language', 'text', 'mining', 'machine', 'learning', 'extract', 'useful', 'information', 'running', 'dogs', 'barking', 'loudly', 'tall', 'buildings']

Before: 41 tokens  |  After: 26 tokens


### Step 6: Stemming
**Stemming** strips suffixes to get the root/base form of a word.
Example: `running → run`, `barking → bark`, `processing → process`

We implement a simple suffix-removal stemmer.

In [6]:
def simple_stem(word):
    # Remove common suffixes (longest match first)
    suffixes = ['tion', 'ing', 'ment', 'ness', 'ity', 'ous', 'ful', 'less', 'er', 'ed', 'ly', 'al']
    for suffix in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[:-len(suffix)]
    return word

stemmed_tokens = [simple_stem(word) for word in filtered_tokens]

print(f'{"Original":<25} Stemmed')
print('-' * 40)
for orig, stem in zip(filtered_tokens, stemmed_tokens):
    print(f'{orig:<25} {stem}')

Original                  Stemmed
----------------------------------------
natural                   natur
language                  language
processing                process
fascinating               fascinat
field                     field
artificial                artifici
intelligence              intelligence
computers                 computers
understand                understand
interpret                 interpret
generate                  generate
human                     human
language                  language
text                      text
mining                    min
machine                   machine
learning                  learn
extract                   extract
useful                    use
information               informa
running                   runn
dogs                      dogs
barking                   bark
loudly                    loud
tall                      tall
buildings                 buildings


### Step 7: Lemmatization
**Lemmatization** converts words to their dictionary base form using vocabulary rules.
It's smarter than stemming — `better → good`, `running → run`, `dogs → dog`

We use a simple lookup dictionary for common words.

In [7]:
# Simple lemmatization dictionary
lemma_dict = {
    'running': 'run', 'barking': 'bark', 'dogs': 'dog',
    'buildings': 'building', 'helps': 'help', 'computers': 'computer',
    'fields': 'field', 'languages': 'language', 'algorithms': 'algorithm',
    'patterns': 'pattern', 'texts': 'text', 'uses': 'use',
    'fascinating': 'fascinate', 'useful': 'use', 'learning': 'learn',
    'processing': 'process', 'mining': 'mine', 'computing': 'compute',
    'extracting': 'extract', 'generating': 'generate', 'interpreting': 'interpret'
}

lemmatized_tokens = [lemma_dict.get(word, word) for word in filtered_tokens]

print(f'{"Original":<25} Lemmatized')
print('-' * 40)
for orig, lemma in zip(filtered_tokens, lemmatized_tokens):
    print(f'{orig:<25} {lemma}')

Original                  Lemmatized
----------------------------------------
natural                   natural
language                  language
processing                process
fascinating               fascinate
field                     field
artificial                artificial
intelligence              intelligence
computers                 computer
understand                understand
interpret                 interpret
generate                  generate
human                     human
language                  language
text                      text
mining                    mine
machine                   machine
learning                  learn
extract                   extract
useful                    use
information               information
running                   run
dogs                      dog
barking                   bark
loudly                    loudly
tall                      tall
buildings                 building


---
## Part 2: TF-IDF

| Term | Formula | Meaning |
|------|---------|--------|
| **TF** | count(word in doc) / total words in doc | How often a word appears in a document |
| **IDF** | log(total docs / docs containing word) | How rare the word is across all documents |
| **TF-IDF** | TF × IDF | Higher score = more important word in that document |

### Step 8: Define the Corpus (Multiple Documents)

In [8]:
corpus = [
    "Natural Language Processing is a field of Artificial Intelligence",
    "Machine learning and text mining extract useful information from data",
    "Deep learning is used in Natural Language Processing and computer vision",
    "Text mining uses machine learning algorithms to find patterns in text"
]

for i, doc in enumerate(corpus, 1):
    print(f'Doc {i}: {doc}')

Doc 1: Natural Language Processing is a field of Artificial Intelligence
Doc 2: Machine learning and text mining extract useful information from data
Doc 3: Deep learning is used in Natural Language Processing and computer vision
Doc 4: Text mining uses machine learning algorithms to find patterns in text


### Step 9: Compute TF-IDF Using Sklearn
`TfidfVectorizer` from sklearn does everything — no extra installation needed.

In [9]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the corpus
tfidf_matrix = vectorizer.fit_transform(corpus)

# Get word names (features)
feature_names = vectorizer.get_feature_names_out()

print(f'Total unique words (features): {len(feature_names)}')
print('Words:', list(feature_names))

Total unique words (features): 28
Words: ['algorithms', 'and', 'artificial', 'computer', 'data', 'deep', 'extract', 'field', 'find', 'from', 'in', 'information', 'intelligence', 'is', 'language', 'learning', 'machine', 'mining', 'natural', 'of', 'patterns', 'processing', 'text', 'to', 'used', 'useful', 'uses', 'vision']


### Step 10: Display TF-IDF Matrix (Documents × Words)

In [10]:
# Convert to pandas DataFrame for easy viewing
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(4),
    index=[f'Doc {i+1}' for i in range(len(corpus))],
    columns=feature_names
)

print('TF-IDF Matrix:')
display(tfidf_df)

TF-IDF Matrix:


,algorithms,and,artificial,computer,data,deep,extract,field,find,from,...,natural,of,patterns,processing,text,to,used,useful,uses,vision
Doc 1,0.0000,0.0000,0.3926,0.0000,0.0000,0.0000,0.0000,0.3926,0.0000,0.0000,...,0.3096,0.3926,0.0000,0.3096,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
Doc 2,0.0000,0.2806,0.0000,0.0000,0.3559,0.0000,0.3559,0.0000,0.0000,0.3559,...,0.0000,0.0000,0.0000,0.0000,0.2806,0.0000,0.0000,0.3559,0.0000,0.0000
Doc 3,0.0000,0.2764,0.0000,0.3506,0.0000,0.3506,0.0000,0.0000,0.0000,0.0000,...,0.2764,0.0000,0.0000,0.2764,0.0000,0.0000,0.3506,0.0000,0.0000,0.3506
Doc 4,0.3201,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3201,0.0000,...,0.0000,0.0000,0.3201,0.0000,0.5048,0.3201,0.0000,0.0000,0.3201,0.0000


### Step 11: Top Important Words Per Document

In [11]:
# For each document, show the top 5 most important words by TF-IDF score
for i, row in tfidf_df.iterrows():
    top_words = row.sort_values(ascending=False).head(5)
    print(f'\n{i} — Top 5 Important Words:')
    for word, score in top_words.items():
        print(f'  {word:<20} {score:.4f}')


Doc 1 — Top 5 Important Words:
  artificial           0.3926
  field                0.3926
  of                   0.3926
  intelligence         0.3926
  language             0.3096

Doc 2 — Top 5 Important Words:
  from                 0.3559
  useful               0.3559
  data                 0.3559
  extract              0.3559
  information          0.3559

Doc 3 — Top 5 Important Words:
  vision               0.3506
  used                 0.3506
  computer             0.3506
  deep                 0.3506
  in                   0.2764

Doc 4 — Top 5 Important Words:
  text                 0.5048
  algorithms           0.3201
  uses                 0.3201
  to                   0.3201
  find                 0.3201
